In [1]:
from ariautils.midi import MidiDict
from ariautils.tokenizer import AbsTokenizer

aria_tokenizer = AbsTokenizer()

midi_file_path = "/mnt/data/improvnet_data/aria-midi-v1-pruned-ext/data/ow/772313_0.mid"
mid = MidiDict.from_midi(midi_file_path)
tokenized_sequence = aria_tokenizer.tokenize(mid)
print(tokenized_sequence[0:100])
print(f"Number of tokens: {len(tokenized_sequence)}")

[('prefix', 'instrument', 'piano'), '<S>', ('piano', 52, 10), ('onset', 0), ('dur', 5000), ('piano', 59, 10), ('onset', 260), ('dur', 5000), ('piano', 64, 20), ('onset', 550), ('dur', 5000), ('piano', 59, 10), ('onset', 820), ('dur', 5000), ('piano', 64, 20), ('onset', 1090), ('dur', 5000), ('piano', 59, 10), ('onset', 1360), ('dur', 4960), ('piano', 52, 20), ('onset', 1650), ('dur', 4670), ('piano', 59, 20), ('onset', 1930), ('dur', 4390), ('piano', 64, 20), ('onset', 2180), ('dur', 4140), ('piano', 59, 10), ('onset', 2450), ('dur', 3870), ('piano', 64, 20), ('onset', 2710), ('dur', 3610), ('piano', 59, 20), ('onset', 2980), ('dur', 3340), ('piano', 40, 20), ('onset', 3290), ('dur', 3030), ('piano', 52, 20), ('onset', 3310), ('dur', 3010), ('piano', 59, 20), ('onset', 3570), ('dur', 2750), ('piano', 64, 20), ('onset', 3830), ('dur', 2490), ('piano', 59, 10), ('onset', 4090), ('dur', 2230), ('piano', 64, 20), ('onset', 4370), ('dur', 1950), ('piano', 59, 20), ('onset', 4620), ('dur', 1

In [ ]:
new_vocab = ()

# Remove all non piano instruments from vocabulary
for token in list(aria_tokenizer.vocab):
    if isinstance(token, tuple) and len(token) == 3:
        if token[0] == "prefix" and token[1] == "instrument" and token[2] != "piano":
            continue
        elif token[0] == "prefix" and token[1] == "composer":
            continue
        elif token[0] == "prefix" and token[1] == "form":
            continue
        elif token[0] == "prefix" and token[1] == "genre":
            continue
        elif token[0] != "piano" and isinstance(token[1], int) and isinstance(token[2], int):
            continue
    new_vocab += (token,)

aria_tokenizer.vocab = new_vocab
vocab_size = len(aria_tokenizer.vocab)
print(f"Vocabulary size: {vocab_size}")

Vocabulary size: 2721


In [2]:
import glob
import os
import json

aria_metadata_filepath = "/mnt/data/improvnet_data/aria-midi-v1-pruned-ext/metadata.json"
with open(aria_metadata_filepath, "r") as f:
    aria_metadata = json.load(f)

# Print a few samples from the config
for key in list(aria_metadata.keys())[:20]:
    print(f"{key}: {aria_metadata[key]}")

# Get all .mid files within multiple directories
aria_midi_files = glob.glob("/mnt/data/improvnet_data/aria-midi-v1-pruned-ext/data/**/*.mid", recursive=True)
print(len(aria_midi_files))

2: {'metadata': {'composer': 'pierné', 'genre': 'classical'}, 'audio_scores': {'0': 0.9721}}
3: {'metadata': {'difficulty': 'advanced'}, 'audio_scores': {'0': 0.9045}}
4: {'metadata': {'composer': 'kondo', 'difficulty': 'advanced', 'music_period': 'modern'}, 'audio_scores': {'0': 0.9757}}
7: {'metadata': {'composer': 'szymanowski', 'opus': 14, 'genre': 'classical', 'form': 'fantasia'}, 'audio_scores': {'0': 0.9647}}
10: {'metadata': {'composer': 'washburn', 'genre': 'classical', 'form': 'waltz', 'difficulty': 'beginner', 'music_period': 'contemporary'}, 'audio_scores': {'0': 0.9394}}
11: {'metadata': {'genre': 'pop'}, 'audio_scores': {'0': 0.9895}}
12: {'metadata': {'composer': 'chabrier', 'genre': 'classical', 'form': 'scherzo', 'piece_number': 10}, 'audio_scores': {'0': 0.719}}
13: {'metadata': {'composer': 'sheeran', 'genre': 'pop', 'music_period': 'modern'}, 'audio_scores': {'0': 0.9875}}
15: {'metadata': {'genre': 'pop', 'music_period': 'modern'}, 'audio_scores': {'0': 0.8136}}
16

In [3]:
aria_data = []
for midi_file in aria_midi_files:
    filename = os.path.basename(midi_file).split("_")[0]
    if filename in aria_metadata:
        entry = {
            "midi_filepath": midi_file,
            "genre": aria_metadata[filename].get('metadata', {}).get('genre', None).lower() if aria_metadata[filename].get('metadata', {}).get('genre', None) else None,
            "composer": aria_metadata[filename].get('metadata', {}).get('composer', None).lower() if aria_metadata[filename].get('metadata', {}).get('composer', None) else None,
            "form": aria_metadata[filename].get('metadata', {}).get('form', None).lower() if aria_metadata[filename].get('metadata', {}).get('form', None) else None,
            "musical_period": aria_metadata[filename].get('metadata', {}).get('musical_period', None).lower() if aria_metadata[filename].get('metadata', {}).get('musical_period', None) else None,
        }
        aria_data.append(entry)

In [4]:
aria_data[0:2]  # Display first two entries

[{'midi_filepath': '/mnt/data/improvnet_data/aria-midi-v1-pruned-ext/data/ro/913085_0.mid',
  'genre': 'classical',
  'composer': 'e',
  'form': None,
  'musical_period': None},
 {'midi_filepath': '/mnt/data/improvnet_data/aria-midi-v1-pruned-ext/data/ro/913706_0.mid',
  'genre': 'classical',
  'composer': 'vierne',
  'form': None,
  'musical_period': None}]

In [5]:
import csv

with open("/mnt/data/improvnet_data/maestro-v3.0.0/maestro-v3.0.0.csv", "r", newline='') as csvfile:
    reader = csv.DictReader(csvfile)
    maestro_metadata = {row['midi_filename']: row for row in reader}    

In [6]:
maestro_data = []
for key, value in maestro_metadata.items():
    midi_filepath = os.path.join("/mnt/data/improvnet_data/maestro-v3.0.0/", value['midi_filename'])
    forms = ['sonata', 'etude', 'waltz', 'nocturne', 'prelude', 'fugue', 'suite', 'ballade', 'mazurka', 'polonaise', 'scherzo', 'fantasy', 'fughetta', 'impromptu', 'variation']
    form = None
    for f in forms:
        if f.lower() in value.get('canonical_title', '').lower():
            form = f.lower()
            break
    entry = {
        "midi_filepath": midi_filepath,
        "genre": "classical",
        "composer": value.get('canonical_composer', None).lower(),
        "form": form,
        "musical_period": None,
    }
    maestro_data.append(entry)

In [7]:
# Get all .mid files within multiple directories
pijama_midi_files = glob.glob("/mnt/data/improvnet_data/pijama-retranscribed/data/**/*.mid", recursive=True)
print(len(pijama_midi_files))

2736


In [8]:
pijama_data = []
for midi_file in pijama_midi_files:
    filename = os.path.basename(midi_file)
    entry = {
        "midi_filepath": midi_file,
        "genre": "jazz",
        "composer": None,
        "form": None,
        "musical_period": None,
    }
    pijama_data.append(entry)

In [ ]:
# Get all .mid files within multiple directories
doug_midi_files = glob.glob("/mnt/data/improvnet_data/doug_mcenzie_jazz/**/*.mid", recursive=True)
print(len(doug_midi_files))

# Remove files -> "My Old FlameGM.mid", "Whilewereyoung.mid"
doug_midi_files = [f for f in doug_midi_files if "My Old FlameGM.mid" not in f and "Whilewereyoung.mid" not in f]

299


In [10]:
doug_data = []
for midi_file in doug_midi_files:
    filename = os.path.basename(midi_file)
    entry = {
        "midi_filepath": midi_file,
        "genre": "jazz",
        "composer": None,
        "form": None,
        "musical_period": None,
    }
    doug_data.append(entry)

In [11]:
all_data = aria_data + maestro_data + pijama_data + doug_data
print(f"Total number of MIDI files collected: {len(all_data)} from {len(aria_data)} (Aria) + {len(maestro_data)} (Maestro) + {len(pijama_data)} (Pijama) + {len(doug_data)} (Doug McKenzie)")

Total number of MIDI files collected: 737000 from 732689 (Aria) + 1276 (Maestro) + 2736 (Pijama) + 299 (Doug McKenzie)


In [12]:
# Get count of all unique genre, composers and form and print them

unique_genres = set()
unique_composers = set()
unique_forms = set()
for entry in all_data:
    if entry['genre']:
        unique_genres.add(entry['genre'])
    if entry['composer']:
        unique_composers.add(entry['composer'])
    if entry['form']:
        unique_forms.add(entry['form'])

print(f"Unique genres: {unique_genres}")
print(f"Unique composers: {unique_composers}")
print(f"Unique forms: {unique_forms}")

Unique genres: {'rock', 'classical', 'ragtime', 'pop', 'blues', 'soundtrack', 'folk', 'atonal', 'ambient', 'jazz'}
Unique composers: {'laub', 'силванский', 'suprana', 'mincarelli', 'pålsson', 'arnes', 'gibb', 'reyes', 'post', 'gasdorf', 'oliinyk', 'lebenson', 'glanvillehicks', 'capasso', 'nystrom', 'wenrich', 'глинка', 'vliegen', 'barbosa', 'francescoli', 'tasman', 'mccuistion', 'yoshihi', 'ponsa', 'saygun', 'koya', 'davitashvili', 'hinchliffe', 'schmid', 'гелер', 'margaritis', 'escande', 'yuyoyuppe', 'faurè', 'sophism', 'ajalyaqin', 'paulus', 'grapsas', 'nizami', 'vitásek', 'petrie', 'tatsuro', 'sarukhanov', 'kalan', 'godsil', 'gasparini', 'naoe', 'alenyev', 'renis', 'ram', 'raimondi', 'krystad', 'lembe', 'helbach', 'prokofjev', 'aballi', 'bolt', 'saint-preux', 'mercadante', 'melrose', 'gil-sok', 'porcaro', 'nobler', 'tchaikovsky', 'ariete', 'blangero', 'gotama', 'xarchakos', 'milet', 'linhai', 'shukh', 'doors', 'demegni', 'lachenmann', 'fedrici', 'clarke', 'moreira', 'parisotti', 'fi

In [13]:
import random

# Write all_data to a JSONL file
with open("/keshav/improvnet_2/improvnet/data/data.jsonl", "w") as f:
    for entry in all_data:
        # Merge forms
        if entry['form'] == "fantasia":
            entry['form'] = "fantasy"
        
        # Train (95%), validation (2%), test split (3%)
        rand_val = random.random()
        if rand_val < 0.95:
            entry['split'] = 'train'
        elif rand_val < 0.97:
            entry['split'] = 'validation'
        else:
            entry['split'] = 'test'

        f.write(json.dumps(entry) + "\n")

## GigaMIDI

In [4]:
import sys
import csv
import os

csv.field_size_limit(sys.maxsize)

csv_filepath = "/mnt/data/improvnet_data/Final_GigaMIDI_V1.1_Final/Final-Metadata-Extended-GigaMIDI-Dataset-updated.csv"

# Read CSV file and print file_path and music_styles_curated keys
with open(csv_filepath, "r") as csvfile:
    reader = csv.DictReader(csvfile)
    # Create data entries for each row
    gigamidi_data = []
    for row in reader:
        # midi_filepath = row['file_path']
        # './Final_GigaMIDI_V1.1_Final/training-V1.1-80%/no-drums/4/81a8984f7cac6fac511e917fe6d307de.mid'
        midi_filepath = os.path.join("/mnt/data/improvnet_data/Final_GigaMIDI_V1.1_Final/", row['file_path'].lstrip("./Final_GigaMIDI_V1.1_Final/"))
        genres = row['music_styles_curated'].lower().split(";") if row['music_styles_curated'] else []
        genre = genres[0] if genres else None
        entry = {
            "midi_filepath": midi_filepath,
            "genre": genre,
            "composer": row['artist'].lower() if row['artist'] else None,
            "form": None,
            "musical_period": None,
        }
        gigamidi_data.append(entry)

print(f"Total number of GigaMIDI MIDI files collected: {len(gigamidi_data)}")

Total number of GigaMIDI MIDI files collected: 2136218


In [5]:
# Count unique genres in gigamidi_data
genre_count = {}
for entry in gigamidi_data:
    genre = entry['genre']
    if genre:
        if genre in genre_count:
            genre_count[genre] += 1
        else:
            genre_count[genre] = 1

# Print genre counts
for genre, count in genre_count.items():
    print(f"{genre}: {count}")

classical: 43916
metal: 1032
game: 21697
world: 297
downtempo: 55
rock: 1956
punk: 642
rap: 146
dance: 92
blues: 66
house: 96
pop: 481
breakbeat: 18
edm: 4
latin: 102
country: 244
alternative: 6
reggae: 86
jazz: 191
trance: 63
disco: 295
techno: 94
soundtrack: 86
folk: 102
drum&bass: 24


In [6]:
gigamidi_data[0:2]  # Display first two entries

[{'midi_filepath': '/mnt/data/improvnet_data/Final_GigaMIDI_V1.1_Final/training-V1.1-80%/no-drums/4/81a8984f7cac6fac511e917fe6d307de.mid',
  'genre': None,
  'composer': None,
  'form': None,
  'musical_period': None},
 {'midi_filepath': '/mnt/data/improvnet_data/Final_GigaMIDI_V1.1_Final/training-V1.1-80%/no-drums/4/d0e3796eb5c2bb37da4fb12433b29949.mid',
  'genre': None,
  'composer': None,
  'form': None,
  'musical_period': None}]

In [7]:
import json
import random

genres = {'rock', 'classical', 'ragtime', 'pop', 'blues', 'soundtrack', 'folk', 'atonal', 'ambient', 'jazz', 'metal', 'game'}

# Write gigamidi_data to a JSONL file and for genres other than the above, set genre to None
with open("/keshav/improvnet_2/improvnet/data/gigamidi_data.jsonl", "w") as f:
    for entry in gigamidi_data:
        if entry['genre'] not in genres:
            entry['genre'] = None
        
        # Train (95%), validation (2%), test split (3%)
        rand_val = random.random()
        if rand_val < 0.95:
            entry['split'] = 'train'
        elif rand_val < 0.97:
            entry['split'] = 'validation'
        else:
            entry['split'] = 'test'

        f.write(json.dumps(entry) + "\n")

## Model

In [3]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.checkpoint import checkpoint
from transformers import PretrainedConfig, PreTrainedModel
import math
from typing import Tuple, Optional, List

# --- Constants ---
# 6 attributes: [instrument, pitch, velocity, onset, duration, seq_index]
NUM_ATTRIBUTES = 6
NUM_VOICE_ATTRIBUTES = 5 
DEFAULT_MRA_BASE_VALUES = [10000.0, 131.0, 20.0, 1031.0, 1031.0, 10000.0]

# --- dLLM-Cache Hyperparameters (Simulated) ---
K_PROMPT_REFRESH = 100 # Kp (Long Interval for 'main' track/control seq)
K_RESPONSE_REFRESH = 6  # Kr (Short Interval for 'accom' track/response)

# --- Configuration ---
class ImprovNetConfig(PretrainedConfig):
    model_type = "improvnet"
    def __init__(
        self,
        hidden_size=72, # Corrected default to 72 for clean math (12 * 6)
        num_heads=12,
        num_layers=2, # Reduced layers for faster test
        ffn_dim=288,  # 72 * 4
        vocab_sizes=[129, 128, 128, 512, 512],
        seq_len=32,   # Reduced seq_len for faster test
        num_genres=10,
        num_forms=5,
        no_bias=False,
        gradient_checkpointing=False,
        initializer_range=0.02,
        adaptive_update_ratio=0.5,
        **kwargs
    ):
        super().__init__(**kwargs)
        self.hidden_size = hidden_size
        self.num_heads = num_heads
        self.num_layers = num_layers
        self.ffn_dim = ffn_dim
        self.vocab_sizes = vocab_sizes
        self.seq_len = seq_len
        self.num_genres = num_genres
        self.num_forms = num_forms
        self.no_bias = no_bias
        self.gradient_checkpointing = gradient_checkpointing
        self.initializer_range = initializer_range
        self.adaptive_update_ratio = adaptive_update_ratio
        
        # We rely on hidden_size being divisible by NUM_ATTRIBUTES for the input layer split
        if hidden_size % NUM_ATTRIBUTES != 0: raise ValueError("hidden_size must be divisible by NUM_ATTRIBUTES for MoonbeamInput")

# --- 1. Input Module (MoonbeamInput) ---
class MoonbeamInput(nn.Module):
    def __init__(self, hidden_size, vocab_sizes, num_dims=NUM_VOICE_ATTRIBUTES):
        super().__init__()
        assert len(vocab_sizes) == num_dims
        self.attr_emb_dim = hidden_size // NUM_ATTRIBUTES # 72 // 6 = 12
        self.output_dim = self.attr_emb_dim * num_dims # 5 * 12 = 60
        self.hidden_size = hidden_size
        
        self.embeds = nn.ModuleList([
            nn.Embedding(vocab_sizes[i], self.attr_emb_dim) for i in range(num_dims)
        ])
        
        # FIX: Project the 60D attribute output to the full 72D hidden size 
        # to match self.pos_embed dimension for element-wise addition.
        self.output_proj = nn.Linear(self.output_dim, self.hidden_size, bias=False)

    def forward(self, instrument, pitch, velocity, onset, duration):
        inputs = [instrument, pitch, velocity, onset, duration]
        embeds = [self.embeds[i](inputs[i]) for i in range(len(inputs))]
        x_5d = torch.cat(embeds, dim=-1) # Shape (B, L, 60)
        
        return self.output_proj(x_5d) # Shape (B, L, 72)

# --- 2. MRA Attention Module (MRANonCausalAttention) ---
class MRANonCausalAttention(nn.Module):
    def __init__(self, config: ImprovNetConfig):
        super().__init__()
        self.config = config
        self.hidden_size = config.hidden_size
        self.num_heads = config.num_heads
        self.head_dim = self.hidden_size // self.num_heads
        self.num_groups = NUM_ATTRIBUTES # 6 groups for 6 attributes

        self.base_values = torch.tensor(DEFAULT_MRA_BASE_VALUES, dtype=torch.float32)

        if self.num_heads % self.num_groups != 0:
            # We assume a fixed heads_per_group and the last group gets the remainder
            self.heads_per_group = self.num_heads // self.num_groups
            # This is complex in a real setting, but for demo, we use the clean split:
            pass
        self.heads_per_group = self.num_heads // self.num_groups


        self.q_proj = nn.Linear(self.hidden_size, self.hidden_size, bias=not config.no_bias)
        self.k_proj = nn.Linear(self.hidden_size, self.hidden_size, bias=not config.no_bias)
        self.v_proj = nn.Linear(self.hidden_size, self.hidden_size, bias=not config.no_bias)
        self.out_proj = nn.Linear(self.hidden_size, self.hidden_size, bias=not config.no_bias)

    # Simplified RoPE rotation function for MRA
    def _apply_rope_rotation(self, tensor: torch.Tensor, position_values: torch.Tensor, base_value: torch.Tensor):
        L, D_h = tensor.shape[2], tensor.shape[3]
        
        freq = 1.0 / (base_value ** (torch.arange(0, D_h, 2, device=tensor.device) / D_h))
        
        # Handle the extra dimension for rotation
        angle = position_values.unsqueeze(1).unsqueeze(-1) * freq
        
        cos = torch.cos(angle).repeat_interleave(2, dim=-1).unsqueeze(1)
        sin = torch.sin(angle).repeat_interleave(2, dim=-1).unsqueeze(1)

        # Apply rotation (complex multiplication approximation)
        q_cos_sin = tensor.clone()
        q_cos_sin[..., 0::2] = tensor[..., 0::2] * cos[..., 0::2] - tensor[..., 1::2] * sin[..., 1::2]
        q_cos_sin[..., 1::2] = tensor[..., 1::2] * cos[..., 0::2] + tensor[..., 0::2] * sin[..., 1::2]

        return q_cos_sin

    # Custom MRA: rotates groups of heads by different attributes
    def _apply_mra(self, q: torch.Tensor, k: torch.Tensor, attributes: torch.Tensor):
        q_rotated, k_rotated = [], []
        
        # Attribute map indices: 0: inst, 1: pitch, 2: vel, 3: onset, 4: duration, 5: seq_index
        # We use the index g itself as the attribute index for simplicity and direct mapping
        
        for g in range(self.num_groups):
            start_head = g * self.heads_per_group
            end_head = (g + 1) * self.heads_per_group
            
            q_group = q[:, :, start_head:end_head, :]
            k_group = k[:, :, start_head:end_head, :]
            
            position_values = attributes[:, :, g].float()
            base_value = self.base_values[g].to(q.device)

            q_rot_group = self._apply_rope_rotation(q_group.permute(0, 2, 1, 3), position_values, base_value).permute(0, 2, 1, 3)
            k_rot_group = self._apply_rope_rotation(k_group.permute(0, 2, 1, 3), position_values, base_value).permute(0, 2, 1, 3)
            
            q_rotated.append(q_rot_group)
            k_rotated.append(k_rot_group)
        
        return torch.cat(q_rotated, dim=2), torch.cat(k_rotated, dim=2)
    
    # Cache format: Tuple[Tensor, Tensor] -> (K_res_cache, V_res_cache)
    def forward(self, hidden_states: torch.Tensor, attributes: torch.Tensor, cache: Optional[Tuple[torch.Tensor, torch.Tensor]] = None, update_ratio: float = 0.0) -> Tuple[torch.Tensor, Tuple[torch.Tensor, torch.Tensor]]:
        B, L_comb, D = hidden_states.shape
        L_main = L_comb // 2 
        
        q = self.q_proj(hidden_states).view(B, L_comb, self.num_heads, self.head_dim)
        k = self.k_proj(hidden_states).view(B, L_comb, self.num_heads, self.head_dim)
        v = self.v_proj(hidden_states).view(B, L_comb, self.num_heads, self.head_dim)
        
        q_main, q_res = q.split(L_main, dim=1)
        k_main, k_res = k.split(L_main, dim=1)
        v_main, v_res = v.split(L_main, dim=1)
        
        k_cached_res, v_cached_res = cache if cache is not None else (None, None)
        
        k_for_attn = k_main
        v_for_attn = v_main
        
        if k_cached_res is not None and v_cached_res is not None:
            if update_ratio > 0.0:
                # V-Verify Mechanism: Select tokens with lowest V similarity
                v_res_flat = v_res.reshape(B * L_main, -1)
                v_cached_flat = v_cached_res.reshape(B * L_main, -1)
                
                similarity = F.cosine_similarity(v_res_flat, v_cached_flat, dim=-1)
                
                k_update = int(L_main * update_ratio)
                _, top_k_indices = torch.topk(similarity, k=k_update, largest=False)
                
                # Adaptive Partial Update for K and V
                k_res_updated = k_cached_res.clone().reshape(B * L_main, -1)
                v_res_updated = v_cached_res.clone().reshape(B * L_main, -1) 
                
                k_res_updated[top_k_indices] = k_res.reshape(B*L_main, -1)[top_k_indices]
                v_res_updated[top_k_indices] = v_res.reshape(B*L_main, -1)[top_k_indices]

                k_for_attn = k_res_updated.reshape(B, L_main, self.num_heads, self.head_dim)
                v_for_attn = v_res_updated.reshape(B, L_main, self.num_heads, self.head_dim)
            else:
                # Pure cache retrieval (update_ratio=0.0)
                k_for_attn = k_cached_res
                v_for_attn = v_cached_res
                
            new_cache = (k_res, v_res) # Store the computed response KV for next step

        else:
            # Initialization or Full Refresh 
            k_for_attn = k_res
            v_for_attn = v_res
            new_cache = (k_res, v_res)
        
        k_comb = torch.cat([k_main, k_for_attn], dim=1)
        v_comb = torch.cat([v_main, v_for_attn], dim=1)
        
        q_rot, k_rot = self._apply_mra(q, k_comb, attributes)

        q_rot = q_rot.permute(0, 2, 1, 3)
        k_rot = k_rot.permute(0, 2, 1, 3)
        v_comb = v_comb.permute(0, 2, 1, 3)
        
        attn_weights = torch.matmul(q_rot, k_rot.transpose(-1, -2)) / (self.head_dim ** 0.5)
        attn_weights = F.softmax(attn_weights, dim=-1)
        
        attn_output = torch.matmul(attn_weights, v_comb)
        attn_output = attn_output.permute(0, 2, 1, 3).contiguous().view(B, L_comb, D)
        output = self.out_proj(attn_output)

        return output, new_cache

# --- 3. Transformer Block (ImprovNetTransformerBlock) ---
# Cache format: Tuple[Tensor, Tensor, Tuple[Tensor, Tensor]] -> (AttnOut_cache, FFNOut_cache, (K_res_cache, V_res_cache))
class ImprovNetTransformerBlock(nn.Module):
    def __init__(self, config: ImprovNetConfig):
        super().__init__()
        self.attn = MRANonCausalAttention(config)
        self.attn_norm = nn.LayerNorm(config.hidden_size)
        
        self.ffn_norm = nn.LayerNorm(config.hidden_size)
        self.ffn_in = nn.Linear(config.hidden_size, config.ffn_dim, bias=not config.no_bias)
        self.ffn_out = nn.Linear(config.ffn_dim, config.hidden_size, bias=not config.no_bias)
        self.ffn_act = nn.GELU()

    def forward(self, hidden_states: torch.Tensor, attributes: torch.Tensor, cache: Optional[Tuple] = None, refresh_prompt: bool = False, refresh_response: bool = False, update_ratio: float = 0.0) -> Tuple[torch.Tensor, Tuple]:
        
        attn_out_cache, ffn_out_cache, kv_cache = (cache[0], cache[1], cache[2]) if cache is not None else (None, None, None)
        
        # --- 1. Attention Block (AttnOut, KV Cached) ---
        attn_input = self.attn_norm(hidden_states)
        
        new_kv_cache = kv_cache
        
        if refresh_prompt or attn_out_cache is None:
            # Full recalculation required for Prompt cache (AttnOut/FFNOut)
            attn_output, new_kv_cache = self.attn(attn_input, attributes, kv_cache, 0.0)
            new_attn_out_cache = attn_output
        elif refresh_response or update_ratio > 0.0:
            # Recompute due to dynamic response or adaptive update
            attn_output, new_kv_cache = self.attn(attn_input, attributes, kv_cache, update_ratio)
            new_attn_out_cache = attn_output
        else:
            # Pure reuse of cached AttnOut
            attn_output = attn_out_cache
            new_attn_out_cache = attn_out_cache
        
        hidden_states = hidden_states + attn_output

        # --- 2. FFN Block (FFNOut Cached) ---
        ffn_input = self.ffn_norm(hidden_states)

        if refresh_prompt or ffn_out_cache is None:
            ffn_output = self.ffn_out(self.ffn_act(self.ffn_in(ffn_input)))
            new_ffn_out_cache = ffn_output
        elif refresh_response or update_ratio > 0.0:
            ffn_output = self.ffn_out(self.ffn_act(self.ffn_in(ffn_input)))
            new_ffn_out_cache = ffn_output
        else:
            ffn_output = ffn_out_cache
            new_ffn_out_cache = ffn_out_cache

        hidden_states = hidden_states + ffn_output
        
        new_cache = (new_attn_out_cache, new_ffn_out_cache, new_kv_cache)
        
        return hidden_states, new_cache

# --- 4. Main Model (ImprovNet) ---
class ImprovNet(PreTrainedModel):
    config_class = ImprovNetConfig
    def __init__(self, config: ImprovNetConfig):
        super().__init__(config)
        self.config = config
        self.seq_len = config.seq_len
        self.vocab_sizes = config.vocab_sizes
        self.gradient_checkpointing = config.gradient_checkpointing
        
        # MoonbeamInput is configured to output the full config.hidden_size (72)
        self.input_embed = MoonbeamInput(config.hidden_size, config.vocab_sizes)
        
        # Positional embedding for the total hidden dimension (72)
        self.pos_embed = nn.Embedding(
            config.seq_len * 2, 
            config.hidden_size
        )
        # Genre and Form embeddings for the full hidden dimension
        self.genre_embed = nn.Embedding(config.num_genres, config.hidden_size)
        self.form_embed = nn.Embedding(config.num_forms, config.hidden_size)
        
        self.transformer_blocks = nn.ModuleList([
            ImprovNetTransformerBlock(config) for _ in range(config.num_layers)
        ])
        self.final_norm = nn.LayerNorm(config.hidden_size)
        
        self.output_heads_main = nn.ModuleList([
            nn.Linear(config.hidden_size, config.vocab_sizes[i], bias=not config.no_bias) 
            for i in range(NUM_VOICE_ATTRIBUTES)
        ])
        self.output_heads_accom = nn.ModuleList([
            nn.Linear(config.hidden_size, config.vocab_sizes[i], bias=not config.no_bias) 
            for i in range(NUM_VOICE_ATTRIBUTES)
        ])
        self.post_init()
        self.pos_embed.weight.data.normal_(mean=0.0, std=config.initializer_range)

    def _init_weights(self, module):
        if isinstance(module, nn.Linear):
            module.weight.data.normal_(mean=0.0, std=self.config.initializer_range)
            if module.bias is not None: module.bias.data.zero_()
        elif isinstance(module, nn.Embedding):
            module.weight.data.normal_(mean=0.0, std=self.config.initializer_range)
            if module.padding_idx is not None: module.weight.data[module.padding_idx].zero_()
        elif isinstance(module, nn.LayerNorm):
            module.bias.data.zero_()
            module.weight.data.fill_(1.0)

    def _calculate_loss(self, all_logits, labels, loss_mask):
        total_loss = 0.0
        loss_count = 0
        loss_fct = nn.CrossEntropyLoss(reduction='none')
        all_labels = torch.unbind(labels, dim=-1)
        all_loss_masks = torch.unbind(loss_mask, dim=-1)
        for i in range(NUM_VOICE_ATTRIBUTES):
            if all_loss_masks[i].any():
                loss_count += 1
                logits_flat = all_logits[i].reshape(-1, self.vocab_sizes[i])
                labels_flat = all_labels[i].reshape(-1)
                mask_flat = all_loss_masks[i].reshape(-1).float()
                raw_loss = loss_fct(logits_flat, labels_flat)
                masked_loss = raw_loss * mask_flat
                attribute_loss = masked_loss.sum() / (mask_flat.sum() + 1e-9)
                total_loss += attribute_loss
        return total_loss, loss_count

    def forward(
        self, 
        input_attributes_main: torch.Tensor,
        input_attributes_accom: torch.Tensor,
        genre: torch.Tensor,
        form: torch.Tensor,
        labels_main: Optional[torch.Tensor] = None,
        labels_accom: Optional[torch.Tensor] = None,
        loss_mask_main: Optional[torch.Tensor] = None,
        loss_mask_accom: Optional[torch.Tensor] = None,
        cache: Optional[List[Tuple]] = None,
        k_step: int = 1,
        return_dict: Optional[bool] = None
    ):
        B, L, A = input_attributes_main.shape
        L_comb = L * 2
        
        return_dict = return_dict if return_dict is not None else self.config.use_return_dict
        
        # Unbind tensors for MoonbeamInput's argument list
        main_attrs_5 = torch.unbind(input_attributes_main, dim=-1)
        accom_attrs_5 = torch.unbind(input_attributes_accom, dim=-1)
        
        # Sequence indices
        seq_indices_main = torch.arange(L, device=self.device).unsqueeze(0).expand(B, L)
        seq_indices_accom = torch.arange(L, 2*L, device=self.device).unsqueeze(0).expand(B, L)

        # 1. Embeddings (now outputting the full hidden_size=72)
        x_main_5 = self.input_embed(*main_attrs_5)
        x_accom_5 = self.input_embed(*accom_attrs_5)
        
        # Positional embedding (full hidden_size=72)
        pos_emb_main = self.pos_embed(seq_indices_main)
        pos_emb_accom = self.pos_embed(seq_indices_accom)
        
        # Combined Input State (Element-wise addition is now 72 + 72)
        x_main = x_main_5 + pos_emb_main
        x_accom = x_accom_5 + pos_emb_accom

        # Genre and Form Embeddings (Conditioning)
        g_emb = self.genre_embed(genre).unsqueeze(1)
        f_emb = self.form_embed(form).unsqueeze(1)
        cond_emb = g_emb + f_emb
        x_main = x_main + cond_emb
        x_accom = x_accom + cond_emb
        
        x_combined = torch.cat([x_main, x_accom], dim=1)
        
        # 2. MRA Position Attributes (6D for rotations)
        attrs_main_6d = torch.cat([input_attributes_main.float(), seq_indices_main.float().unsqueeze(-1)], dim=-1)
        attrs_accom_6d = torch.cat([input_attributes_accom.float(), seq_indices_accom.float().unsqueeze(-1)], dim=-1)
        attrs_combined = torch.cat([attrs_main_6d, attrs_accom_6d], dim=1)

        # 3. Caching Management Logic
        is_initialization = (k_step == 0)
        k = k_step
        refresh_prompt = (k % K_PROMPT_REFRESH == 0)
        refresh_response = (k % K_RESPONSE_REFRESH == 0)

        adaptive_update_ratio = self.config.adaptive_update_ratio if not refresh_response and not refresh_prompt else 0.0

        new_cache_list = []
        current_cache_list = cache if cache is not None and not is_initialization else [None] * self.config.num_layers
        
        use_cache = not self.training
        
        # Transformer Blocks Loop
        for i, block in enumerate(self.transformer_blocks):
            prev_layer_cache = current_cache_list[i]

            if self.gradient_checkpointing and self.training:
                output_tuple = checkpoint(block, x_combined, attrs_combined, None, False, False, 0.0, use_reentrant=False)
                x_combined = output_tuple[0]
                new_cache_list.append(None)
            else:
                x_combined, new_block_cache = block(
                    hidden_states=x_combined, 
                    attributes=attrs_combined, 
                    cache=prev_layer_cache, 
                    refresh_prompt=refresh_prompt, 
                    refresh_response=refresh_response, 
                    update_ratio=adaptive_update_ratio
                )
                new_cache_list.append(new_block_cache)
                
        # 4. Final Norm, Logits, and Loss
        x_combined_norm = self.final_norm(x_combined)
        x_main_norm, x_accom_norm = x_combined_norm.chunk(2, dim=1)
        
        logits_main = tuple(head(x_main_norm) for head in self.output_heads_main)
        logits_accom = tuple(head(x_accom_norm) for head in self.output_heads_accom)
        
        total_loss = None
        if labels_main is not None:
            loss_main, count_main = self._calculate_loss(logits_main, labels_main, loss_mask_main)
            loss_accom, count_accom = self._calculate_loss(logits_accom, labels_accom, loss_mask_accom)
            total_loss_value = loss_main + loss_accom
            total_count = count_main + count_accom
            total_loss = total_loss_value / total_count if total_count > 0 else torch.tensor(0.0, device=x_combined_norm.device)
        
        output = {"logits_main": logits_main, "logits_accom": logits_accom}
        if total_loss is not None: output["loss"] = total_loss
        if use_cache: output["cache"] = new_cache_list
        
        if not return_dict:
            list_output = []
            if total_loss is not None: list_output.append(output["loss"])
            list_output.extend([output["logits_main"], output["logits_accom"]])
            if use_cache: list_output.append(output["cache"])
            return tuple(list_output)
        
        return output

# --- Dummy Test ---
def run_dummy_test():
    # Model parameters for test
    SEQ_LEN = 32
    HIDDEN_SIZE = 72
    VOCAB_SIZES = [129, 128, 128, 512, 512] 
    
    config = ImprovNetConfig(
        hidden_size=HIDDEN_SIZE,
        num_heads=12,
        num_layers=2,
        ffn_dim=HIDDEN_SIZE * 4,
        vocab_sizes=VOCAB_SIZES,
        seq_len=SEQ_LEN,
        adaptive_update_ratio=0.5
    )
    model = ImprovNet(config)
    model.eval()

    B = 1
    device = model.device
    
    # Corrected random data generation (generates input within vocab bounds)
    input_main_list = []
    input_accom_list = []
    for vocab_size in VOCAB_SIZES:
        rand_main = torch.randint(0, vocab_size, (B, SEQ_LEN), device=device)
        rand_accom = torch.randint(0, vocab_size, (B, SEQ_LEN), device=device)
        input_main_list.append(rand_main)
        input_accom_list.append(rand_accom)

    input_main = torch.stack(input_main_list, dim=-1)
    input_accom = torch.stack(input_accom_list, dim=-1)
    genre = torch.tensor([0], device=device)
    form = torch.tensor([0], device=device)

    # --- Step 1: Initialization (k_step=0, Full Compute/Cache) ---
    print(f"--- Step 1 (k=0): Initialization (Full Compute/Cache) ---")
    output1 = model(input_main, input_accom, genre, form, k_step=0, return_dict=True)
    cache1 = output1['cache']
    
    print(f"Cache size: {len(cache1)} layers. First KV cache is non-None: {cache1[0][2] is not None}")
    
    # --- Step 2: Adaptive Update (k_step=1) ---
    print(f"\n--- Step 2 (k=1): Adaptive Update (Ratio={config.adaptive_update_ratio}) ---")
    output2 = model(input_main, input_accom, genre, form, cache=cache1, k_step=1, return_dict=True)
    
    logits_main_shape = output2['logits_main'][0].shape
    print(f"Logits Main Shape: {logits_main_shape} (Expected: B x L x Vocab)")
    
    # --- Step 3: Response Full Refresh (k_step=K_RESPONSE_REFRESH) ---
    print(f"\n--- Step 3 (k={K_RESPONSE_REFRESH}): Response Full Refresh ---")
    output3 = model(input_main, input_accom, genre, form, cache=output2['cache'], k_step=K_RESPONSE_REFRESH, return_dict=True)

    print("\nDummy test successful! No dimension errors encountered.")
    return model

if __name__ == '__main__':
    run_dummy_test()

--- Step 1 (k=0): Initialization (Full Compute/Cache) ---
Cache size: 2 layers. First KV cache is non-None: True

--- Step 2 (k=1): Adaptive Update (Ratio=0.5) ---
Logits Main Shape: torch.Size([1, 32, 129]) (Expected: B x L x Vocab)

--- Step 3 (k=6): Response Full Refresh ---

Dummy test successful! No dimension errors encountered.
